In [1]:
import pandas as pd

In [2]:
dataset=pd.read_csv("Tatacoffee13_21.csv")

In [3]:
dataset

,Date,Open,High,Low,Close
0,2013-01-01,1410.60,1427.90,1408.30,1415.10
1,2013-01-02,1421.00,1626.60,1416.15,1607.40
2,2013-01-03,1632.55,1673.90,1613.05,1626.20
3,2013-01-04,1627.75,1627.75,1574.60,1579.05
4,2013-01-07,1580.00,1639.50,1565.50,1595.65
...,...,...,...,...,...
2220,2021-12-22,202.90,207.80,201.35,205.00
2221,2021-12-23,206.00,206.85,202.05,202.95
2222,2021-12-24,203.90,203.90,199.35,201.00
2223,2021-12-27,200.00,222.00,196.00,218.35


In [4]:
dataset["Open"].value_counts()

Open
91.00     11
90.50     10
88.00      9
92.50      9
91.50      9
          ..
208.00     1
196.00     1
202.90     1
203.90     1
219.65     1
Name: count, Length: 1401, dtype: int64

In [5]:
dataset.isnull().sum()

Date     0
Open     0
High     0
Low      0
Close    0
dtype: int64

In [7]:
data_train=dataset.head(1500)

In [9]:
data_test=dataset.tail(509)

In [19]:
timedata=pd.DataFrame()

In [20]:
timedata["Date"]=dataset["Date"]
timedata["Close"]=dataset["Close"]

In [21]:
data=timedata["Close"]

In [24]:
import numpy as np
orders=[(1,0,2),(1,0,1),(2,0,1),(1,0,1)]

orderslist=[]
rscorelist=[]

for i in orders:

    orderslist.append(i)

    # Corrected import: SARIMAX is in statsmodels.tsa.statespace.sarimax, not statsmodels.tsa.arima.model
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    model = SARIMAX(data, order=i)
    model_fit = model.fit()

    y_pred = model_fit.predict(0, len(data)-1)

    from sklearn.metrics import r2_score
    r2 = r2_score(data, y_pred)

    rscorelist.append(r2)

    print("Order={} ,r2={}".format(i,r2))

Order=(1, 0, 2) ,r2=0.9923915926931723
Order=(1, 0, 1) ,r2=0.9923911256946636
Order=(2, 0, 1) ,r2=0.9923914443547096
Order=(1, 0, 1) ,r2=0.9923911256946636


In [25]:
import warnings
warnings.filterwarnings("ignore")

In [34]:
trends=['n','t','c','ct']
orders=[(0,0,0),(0,0,1),(2,0,1),(1,1,1)]
orderslist=[]
trendslist=[]
rscorelist=[]

# Data preprocessing to fix the ValueError
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import r2_score

# Define sample data since data_train and data_test were not defined
# Replace this with your actual data loading
np.random.seed(42)  # For reproducible results
data_train = pd.Series(np.random.randn(100).cumsum())  # Sample time series data
data_test = pd.Series(np.random.randn(20).cumsum())    # Sample test data

# Ensure data is numeric and handle missing values
data_train = pd.to_numeric(data_train, errors='coerce')  # Convert to numeric, NaN for non-numeric
data_test = pd.to_numeric(data_test, errors='coerce')

# Remove or fill missing values
data_train = data_train.dropna()  
data_test = data_test.dropna()    

# Ensure data is float type
data_train = data_train.astype(float)
data_test = data_test.astype(float)  # Fixed variable name from datas_test

for td in trends:
    for i in orders:
        trendslist.append(td)
        orderslist.append(i)
        
        try:  # Add error handling for model fitting
            model = SARIMAX(data_train, order=i, seasonal_order=(0,0,0,0), trend=td)  # Fixed variable name
            model_fit = model.fit(disp=False)  # disp=False to suppress convergence messages

            y_pred = model_fit.predict(start=len(data_train), end=len(data_train)+len(data_test)-1)  # Fixed variable name

            r2 = r2_score(data_test, y_pred)

            rscorelist.append(r2)
            print("Trend={}, Order={}, r2={}".format(td,i,r2))
            
        except Exception as e:  # Handle any model fitting errors
            print("Error with Trend={}, Order={}: {}".format(td,i,str(e)))
            rscorelist.append(np.nan)  # Add NaN for failed models

Trend=n, Order=(0, 0, 0), r2=-0.9502111323547064
Trend=n, Order=(0, 0, 1), r2=-1.2200744351302117
Trend=n, Order=(2, 0, 1), r2=-47.73326345077934
Trend=n, Order=(1, 1, 1), r2=-57.55679445903328
Trend=t, Order=(0, 0, 0), r2=-115.31445722142917
Trend=t, Order=(0, 0, 1), r2=-114.01453676704283
Trend=t, Order=(2, 0, 1), r2=-94.36841126635089
Trend=t, Order=(1, 1, 1), r2=-81.81252436453232
Trend=c, Order=(0, 0, 0), r2=-18.564825658678274
Trend=c, Order=(0, 0, 1), r2=-19.249885077569104
Trend=c, Order=(2, 0, 1), r2=-47.75182720819125
Trend=c, Order=(1, 1, 1), r2=-74.09073882498845
Trend=ct, Order=(0, 0, 0), r2=-114.61341805801298
Trend=ct, Order=(0, 0, 1), r2=-113.24084922771739
Trend=ct, Order=(2, 0, 1), r2=-90.03257139871073
Trend=ct, Order=(1, 1, 1), r2=-51.99661520707277


In [35]:
rscorelist

[-0.9502111323547064,
 -1.2200744351302117,
 -47.73326345077934,
 -57.55679445903328,
 -115.31445722142917,
 -114.01453676704283,
 -94.36841126635089,
 -81.81252436453232,
 -18.564825658678274,
 -19.249885077569104,
 -47.75182720819125,
 -74.09073882498845,
 -114.61341805801298,
 -113.24084922771739,
 -90.03257139871073,
 -51.99661520707277]

In [36]:
result=pd.DataFrame()

In [37]:
result.insert(0,"Trend",trendslist)
result.insert(1,"Order",orderslist)
result.insert(2,"R_score",rscorelist)

result

,Trend,Order,R_score
0,n,"(0, 0, 0)",-0.950211
1,n,"(0, 0, 1)",-1.220074
2,n,"(2, 0, 1)",-47.733263
3,n,"(1, 1, 1)",-57.556794
4,t,"(0, 0, 0)",-115.314457
5,t,"(0, 0, 1)",-114.014537
6,t,"(2, 0, 1)",-94.368411
7,t,"(1, 1, 1)",-81.812524
8,c,"(0, 0, 0)",-18.564826
9,c,"(0, 0, 1)",-19.249885


In [38]:
model_fit.predict(len(data), len(data)+5)

2225    5587.412589
2226    5592.664877
2227    5597.919629
2228    5603.176843
2229    5608.436521
2230    5613.698661
Name: predicted_mean, dtype: float64